In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.sklearn.processing import ScriptProcessor
from sagemaker.processing import ProcessingStep
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import CacheConfig

In [ ]:
# 1. Configuración inicial
role = get_execution_role()
sagemaker_session = sagemaker.Session()
default_bucket = sagemaker_session.default_bucket()

# URI del script en S3
uri_script = f"s3://{default_bucket}/scripts/PEst_EC/TEST_PE_EC_1_estrategico.py"

# --- PASO ÚNICO: PEDIDO ESTRATÉGICO ---
# Este script genera el PE, excluye PR y PS, concatena los 3 y sube al bucket de orders
processor = ScriptProcessor(
    command=['python3'],
    image_uri=sagemaker.image_uris.retrieve("sklearn", sagemaker_session.boto_region_name, "1.2-1"),
    role=role,
    instance_count=1,
    instance_type="ml.m5.2xlarge"
)

step_estrategico = ProcessingStep(
    name="PE-Ecuador-Estrategico",
    processor=processor,
    code=uri_script
)

In [ ]:
# --- DEFINICIÓN Y CREACIÓN DEL PIPELINE ---
pipeline = Pipeline(
    name="Pipeline-PedidoEstrategico-Ecuador",
    steps=[step_estrategico],
    sagemaker_session=sagemaker_session
)

pipeline.upsert(role_arn=role)
print("Pipeline registrado con éxito.")

In [ ]:
# Descomenta para ejecutar manualmente:
# execution = pipeline.start()
# execution.wait()